In [ ]:
import pandas as pd
from pathlib import Path
import re

# Paths
folder_path = Path('./bikeshare_data/tracts_with_trips')
out_path    = Path('./correlation_results_2')
out_path.mkdir(parents=True, exist_ok=True)

census_files = [
    './census_data/S0101_2023.csv',
    './census_data/S0601_2023.csv',
    './census_data/S0701_2023.csv',
    './census_data/S0801_2023.csv',
    './census_data/S0802_2023.csv',
    './census_data/DP03_2023.csv'
]

# Load & clean ACS tables  (keep only *_E estimate columns)
def load_clean_acs(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)

    if 'GEOID' in df.columns:
        df['GEOID'] = df['GEOID'].str[-11:]
    elif 'GEO_ID' in df.columns:
        df['GEOID'] = df['GEO_ID'].str[-11:]
    else:
        raise ValueError(f'No GEOID in {path}')
    
    df = df.drop(columns=[c for c in df.columns if c.upper() == 'NAME'])

    keep_cols = ['GEOID'] + [c for c in df.columns if re.search(r'E$', c)]
    df = df[keep_cols]

    for col in df.columns.difference(['GEOID']):
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

census_tables = [load_clean_acs(f) for f in census_files]

census_all = census_tables[0]
for tbl in census_tables[1:]:
    census_all = census_all.merge(tbl, on='GEOID', how='outer')

numeric_census_cols = census_all.columns.difference(['GEOID'])

# Loop through every city, compute Pearson & Spearman, save
for city_file in folder_path.glob('*_tracts_with_trips.csv'):
    city_name = city_file.stem.split('_')[0]          # e.g. "chicago"
    city_df   = pd.read_csv(city_file, dtype={'GEOID': str})
    city_df['GEOID'] = city_df['GEOID'].str[-11:]

    merged = city_df.merge(census_all, on='GEOID', how='left')

    cols_to_corr = numeric_census_cols.intersection(merged.columns).tolist() + ['trips_per_day']
    corr_df      = merged[cols_to_corr]

    # Pearson
    pearson = corr_df.corr(method='pearson')['trips_per_day'] \
                     .drop('trips_per_day') \
                     .dropna()

    # --- Spearman (rank / “regular”) --------------------------------
    spearman = corr_df.corr(method='spearman')['trips_per_day'] \
                      .drop('trips_per_day') \
                      .reindex(pearson.index)        # align rows

    # combine and write
    out = pd.concat([pearson, spearman], axis=1)
    out.columns = ['pearson_r', 'spearman_r']
    out.to_csv(out_path / f'{city_name}_census_vs_trips_corr.csv')

    print(f'{city_name}: saved {out.shape[0]} vars → '
          f'{out_path / f"{city_name}_census_vs_trips_corr.csv"}')

In [ ]:
files =['./multiple_regression/austin_census_data_and_trips.csv',
        './multiple_regression/nyc_census_data_and_trips.csv',
        './multiple_regression/oh_census_data_and_trips.csv',
        './multiple_regression/sf_census_data_and_trips.csv']

combined_data = []

import os

for file in files:
    # Extract just the file name first
    base_name = os.path.basename(file)
    # Extract city name from the filename
    city_name = base_name.split('_')[0]  # Assumes 'city_census_data_and_trips.csv' format
    # Load the data
    city_data = pd.read_csv(file)
    
    # Add a column for the city name
    city_data['city'] = city_name

    # Append to the combined data list
    combined_data.append(city_data)

# Combine all city data into a single DataFrame
combined_df = pd.concat(combined_data, ignore_index=True)

# Save the combined DataFrame to a CSV
combined_df.to_csv('./multiple_regression/all_census_data_and_trips.csv', index=False)
print(combined_df.shape)